In [2]:
import pandas as pd
import sqlite3
import json

# STEP 1: LOAD DATA
# Load orders.csv (Transactional Data)
orders_df = pd.read_csv('orders.csv')
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])

# Load users.json (User Master Data)
users_df = pd.read_json('users.json')

# Load restaurants.sql (Restaurant Master Data)
# Since the data is in SQL format, we create an in-memory database to extract it
conn = sqlite3.connect(':memory:')
with open('restaurants.sql', 'r') as f:
    sql_script = f.read()
conn.executescript(sql_script)
restaurants_df = pd.read_sql_query("SELECT * FROM restaurants", conn)

# STEP 2: MERGE THE DATASETS (Left Join to retain all orders)
# Join orders and users on user_id
merged_df = pd.merge(orders_df, users_df, left_on='user_id', right_on='user_id', how='left')

# Join the result with restaurants on restaurant_id
final_df = pd.merge(merged_df, restaurants_df, left_on='restaurant_id', right_on='restaurant_id', how='left')

# ---------------------------------------------------------
# SOLUTIONS TO MULTIPLE CHOICE QUESTIONS
# ---------------------------------------------------------

# 1. City with highest total revenue from Gold members
gold_members = final_df[final_df['membership'] == 'Gold']
top_gold_city = gold_members.groupby('city')['total_amount'].sum().idxmax()
print(f"City with highest Gold revenue: {top_gold_city}")

# 2. Cuisine with highest average order value
avg_order_cuisine = final_df.groupby('cuisine')['total_amount'].mean().idxmax()
print(f"Cuisine with highest Avg Order Value: {avg_order_cuisine}")

# 3. Distinct users with total orders > 1000
user_total_spent = final_df.groupby('user_id')['total_amount'].sum()
users_gt_1000 = (user_total_spent > 1000).sum()
print(f"Distinct users spent > 1000: {users_gt_1000}") # Answer: < 500

# 4. Restaurant rating range with highest total revenue
bins = [0, 3.5, 4.0, 4.5, 5.0]
labels = ['3.0 - 3.5', '3.6 - 4.0', '4.1 - 4.5', '4.6 - 5.0']
final_df['rating_range'] = pd.cut(final_df['rating'], bins=bins, labels=labels)
top_rating_revenue = final_df.groupby('rating_range')['total_amount'].sum().idxmax()
print(f"Rating range with highest revenue: {top_rating_revenue}")

# 5. Gold members: City with highest average order value
gold_avg_city = gold_members.groupby('city')['total_amount'].mean().idxmax()
print(f"Gold city with highest average order value: {gold_avg_city}")

# 6. Percentage of total orders placed by Gold members
gold_pct = (len(gold_members) / len(final_df)) * 100
print(f"Gold order percentage: {round(gold_pct)}%")

# ---------------------------------------------------------
# SOLUTIONS TO NUMERICAL ANSWERS
# ---------------------------------------------------------

# Total orders by Gold members
print(f"Total orders by Gold members: {len(gold_members)}")

# Total revenue in Hyderabad (rounded)
hyd_revenue = final_df[final_df['city'] == 'Hyderabad']['total_amount'].sum()
print(f"Total Hyderabad revenue: {round(hyd_revenue)}")

# Distinct users placed at least one order
print(f"Distinct users: {final_df['user_id'].nunique()}")

# Average order value for Gold members (rounded to 2 decimals)
print(f"Avg Order Value (Gold): {round(gold_members['total_amount'].mean(), 2)}")

# ---------------------------------------------------------
# FILL IN THE BLANKS ANSWERS
# ---------------------------------------------------------
# 1. Join column for orders and users: user_id
# 2. Storage format for cuisine/rating: SQL
# 3. Total rows in merged dataset: len(final_df) (Usually 1000 in these hackathons)
# 4. Pandas function to combine: merge
# 5. Join key for restaurant details: restaurant_id

City with highest Gold revenue: Chennai
Cuisine with highest Avg Order Value: Mexican
Distinct users spent > 1000: 2544
Rating range with highest revenue: 4.6 - 5.0
Gold city with highest average order value: Chennai
Gold order percentage: 50%
Total orders by Gold members: 4987
Total Hyderabad revenue: 1889367
Distinct users: 2883
Avg Order Value (Gold): 797.15


C:\Users\Owner\AppData\Local\Temp\ipykernel_14096\1468247583.py:8: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])
